# Addis Ababa Road Traffic Accident Analysis
## Stage 3: Data Quality, Integrity, and Trustworthiness Audit

**Author:** YonInsights (`yo2120018@gmail.com`)  
**Kernel:** Python (Addis Ababa Road Safety)

---

### 1. Research Objective
Before performing statistical analysis or training machine learning models, we must assess whether the data can be trusted. This audit evaluates:
1. Exact duplicate records across all 31 variables.
2. Explicit missing values (`NaN`) vs. implicit pseudo-missing values (`'na'`, `'Unknown'`).
3. Numeric ranges and plausibility of extreme values (outliers).
4. Categorical anomalies and encoding artifacts (e.g. `'7'` and `'5'` in vehicle defects).
5. Scientific justification for cleaning decisions (preserving raw data).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# 1. Load raw data
DATA_PATH = Path("../Data/Addis_Ababa_city_RTA.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("Data/Addis_Ababa_city_RTA.csv")

df_raw = pd.read_csv(DATA_PATH)
print(f"Raw dataset loaded: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns.")

In [ ]:
# 2. Duplicate Check across all 31 columns
duplicate_count = df_raw.duplicated().sum()
print(f"Exact duplicate rows: {duplicate_count}")

In [ ]:
# 3. Missing Value Audit: Explicit (NaN) vs. Implicit ('na', 'Unknown')
explicit_nan = df_raw.isna().sum()

# Identify implicit text tokens used as missing markers
def count_pseudo_na(series):
    if series.dtype == 'object' or str(series.dtype).startswith('str'):
        return series.astype(str).str.strip().isin(['na', 'Unknown', 'unknown', 'None', '?']).sum()
    return 0

implicit_na = df_raw.apply(count_pseudo_na)
total_missing = explicit_nan + implicit_na

quality_df = pd.DataFrame({
    'Explicit_NaN': explicit_nan,
    'Explicit_%': (explicit_nan / len(df_raw) * 100).round(2),
    'Implicit_Text_NA': implicit_na,
    'Implicit_%': (implicit_na / len(df_raw) * 100).round(2),
    'Total_Effective_Missing': total_missing,
    'Total_Effective_%': (total_missing / len(df_raw) * 100).round(2)
})

# Display top columns affected by missingness
quality_df[quality_df['Total_Effective_Missing'] > 0].sort_values(by='Total_Effective_%', ascending=False)

In [ ]:
# 4. Numeric Column Range & Plausibility Verification
numeric_summary = df_raw[['Number_of_vehicles_involved', 'Number_of_casualties']].describe().T
numeric_summary['IQR'] = numeric_summary['75%'] - numeric_summary['25%']
numeric_summary

In [ ]:
# 5. Categorical Encoding Artifacts: Defect_of_vehicle
print("Defect_of_vehicle frequency distribution:")
print(df_raw['Defect_of_vehicle'].value_counts(dropna=False))